# Projet : Stage
### Objectif

Pour les attributs spatiaux et temporels, on voudrait faire la même chose. Pour cette partie, il y a 3 étapes à faire : 

1. Identifier les attributs spatiaux / temporels. 
2. Trouver le niveau d'hiérarchie de chaque attributs selon les hiérarchies spatiales / temporelles
3. Identifier le niveau d'hiérarchie le plus fin parmis tous les attributs spatiaux / temporels en tant que granularité minimum de dataset; identifier l'attribut au niveau d'hiérarchie le plus haut parmis tous les attributs en tant que scope de dataset et donner la liste de ses valeurs distinctes. 

L'output final qu'on demande est un dossier json de métadonnée de tous les datasets.

## 1. Hiérarchisation des données

In [1]:
import json
import csv

def construire_dictionnaire_hierarchise():

    def fillChamp(dicChamps, dic_hierarchise, rang):
        for i in range(len(dicChamps['features'])):
            champ = dicChamps['features'][i]['properties']['nom']
            if champ not in dic_hierarchise[rang]:
                dic_hierarchise[rang].append(champ.lower())
        return

    def fillDictionnaireGeoJSON():
        fichiers = ['communes', 'departements', 'regions']
        dic_hierarchise = {}

        for fichier in fichiers:
            dic_hierarchise[fichier] = []
            with open(f"levels/france-geojson/{fichier}-avec-outre-mer.geojson", "r", encoding="utf-8") as mon_json:
                data = json.load(mon_json)
                fillChamp(data, dic_hierarchise, fichier)

        return dic_hierarchise

    def fillDictionnaireQuartiers(dic_hierarchise):
        with open("liste-correspondance-qp2024-qp2015.csv", "r", encoding="utf-8") as fichier:
            reader = csv.reader(fichier, delimiter=";")
            listeQuartiers = list(reader)[1:]  # Ignorer l'en-tête

        dic_hierarchise['quartiers'] = []
        for i in range(len(listeQuartiers)):
            quartier = listeQuartiers[i][1]
            if quartier not in dic_hierarchise['quartiers'] and quartier != "":
                dic_hierarchise['quartiers'].append(quartier.lower())

        dic_hierarchise['QP'] = []
        for i in range(len(listeQuartiers)):
            quartier = listeQuartiers[i][3]
            if quartier not in dic_hierarchise['QP'] and quartier != "":
                dic_hierarchise['quartiers'].append(quartier.lower())

    dic_hierarchise = fillDictionnaireGeoJSON()
    fillDictionnaireQuartiers(dic_hierarchise)    

    # Tri des listes dans le dictionnaire
    champs = ['regions', 'departements', 'communes', 'cantons', 'quartiers', 'QP']
    dic_hierarchise = {champ: dic_hierarchise[champ] for champ in champs if champ in dic_hierarchise}
    dic_hierarchise['pays'] = ['france', 'france métropolitaine', 'france d\'outre-mer', 'france entière']
    
    return dic_hierarchise

def recuperer_dictionnaire_hierarchise():
    try:
        with open("dic_hierarchise.json", "r", encoding="utf-8") as fichier:
            dic_hierarchise = json.load(fichier)
    except FileNotFoundError:
        dic_hierarchise = construire_dictionnaire_hierarchise()
        with open("dic_hierarchise.json", "w", encoding="utf-8") as fichier:
            json.dump(dic_hierarchise, fichier, ensure_ascii=False, indent=4)
    
    return dic_hierarchise

In [9]:
# Appel de la fonction pour obtenir le dictionnaire hiérarchisé
dic_hierarchise = recuperer_dictionnaire_hierarchise()
# champs_ranges = ['regions', 'departements', 'communes', 'cantons', 'quartiers', 'QP', 'geopoint']
champs_ranges = ['pays', 'regions', 'departements', 'quartiers', 'communes', 'iris', 'geopoints']
temps_ranges = ['annee', 'trimestre', 'mois', 'semaine', 'date']
hierarchie_champs_spa = {champ: (len(champs_ranges) - i) for i, champ in enumerate(champs_ranges)}
hierarchie_temps_spa = {temps: (len(temps_ranges) - i) for i, temps in enumerate(temps_ranges)}

## 2. Identification des attributs spatiaux dans un fichier csv/xlsx

### 2.1 Récupérer tous les attributs spatiaux

#### 2.1.1 Recuperation de tous les datasets

In [4]:
import os

def getFiles(origine='Opendata'):
    fichiers = []

    def separateurFichier(datasets):
        datasetSepares = {}

        for dataset in datasets:
            if dataset.endswith('.csv'):
                if 'csv' not in datasetSepares:
                    datasetSepares['csv'] = []
                datasetSepares['csv'].append(dataset)

            elif dataset.endswith('.xlsx'):
                if 'xlsx' not in datasetSepares:
                    datasetSepares['xlsx'] = []
                datasetSepares['xlsx'].append(dataset)

        return datasetSepares

    for dossier in os.walk(origine):
        for fichier in dossier[2]:
            if fichier.endswith('.csv') or fichier.endswith('.xlsx'):
                fichiers.append(os.path.join(dossier[0], fichier))
    
    return separateurFichier(fichiers)

datasets = getFiles()

#### 2.1.2 Recherche des attributs spatiaux via contenu des cellules

Variables utiles

In [ ]:
import re

regexAnnee = r'(19\d{2}|20\d{2})$'
regexMois = r'(0[1-9]|1[0-2])'
regexJour = r'(0[1-9]|[12]\d|3[01])'
regexDate = r'(' + regexAnnee[:-1] + r'[-/]' + regexMois + r'[-/]' + regexJour + r')'
regexHeure = r'([01]\d|2[0-3]):([0-5]\d):([0-5]\d)'
regexTrim = r'(19\d{2}|20\d{2})_[a-zA-Z]{1}[1-3]'
regexGeopoint1 = r'^(-?\d+(?:\.\d+)?)[,; ]\s*(-?\d+(?:\.\d+)?)$'
regexGeopoint2 = r'^(-?\d+(?:\.\d+)?)\s+(-?\d+(?:\.\d+)?)$'

listeRegexTemporel = [[regexDate, 'date'], [regexHeure, 'heure'], [regexAnnee, 'annee'], [regexTrim, 'trimestre']]
listeRegexGeopoint = [[regexGeopoint1, 'geopoints'], [regexGeopoint2, 'geopoints']]

Fonctions utiles

In [ ]:
import re
import pandas as pd

def estLatitude(val):
    try:
        v = float(val)
        return -90 <= v <= 90
    except:
        return False

def estLongitude(val):
    try:
        v = float(val)
        return -180 <= v <= 180
    except:
        return False

def estGeopoint(attribut):
    if isinstance(attribut, str):
        attribut = attribut.strip()
            
        for regex in listeRegexGeopoint:
            if not isinstance(attribut, float):
                attribut = str(attribut)
                if re.match(r'^'+regex[0]+'$', attribut):
                    lat, lon = map(float, attribut.split(','))
                    if -90 <= lat <= 90 and -180 <= lon <= 180:
                        return [True, regex[1]]
                elif estLatitude(attribut) or estLongitude(attribut):
                    return [True, regex[1]]
    
    return [False, None]

listeIRIS = pd.read_csv('table_passage_1999_2022.csv', sep=',', encoding='utf-8')
headersIRIS = listeIRIS.columns.tolist()

def estIRIS(attribut):
    for i in range(len(headersIRIS)):
        for j in range(len(listeIRIS)):
            cell = listeIRIS.iloc[j, i]
            if attribut == cell:
                return [True, 'iris']
    return [False, None]

def estSpatial(attribut):
    for champ, valeurs in dic_hierarchise.items():
        if attribut in valeurs:
            return [True, champ]
        infoGeopoint = estGeopoint(attribut)
        if infoGeopoint[0]:
            return infoGeopoint
        infoIRIS = estIRIS(attribut)
        if infoIRIS[0]:
            return infoIRIS
    return [False, None]

def estTemporel(attribut):
    for regex in listeRegexTemporel:
        if not isinstance(attribut, str):
            attribut = str(attribut)
        if re.match(regex[0], attribut):
            return [True, regex[1]]
        elif attribut in ['janvier', 'fevrier', 'mars', 'avril', 'mai', 'juin', 'juillet', 'aout', 'septembre', 'octobre', 'novembre', 'decembre']:
            return [True, 'mois']
    return [False, None]

def recupererAttributsSpatiaux(headers, df, score_colonne):
    liste_attributs_spatiaux = {}
    for i in range(10):
        for j in range(len(headers)):
            cell = df.iloc[i, j]
            if isinstance(cell, str):
                cell = cell.lower()

            infoSpatial = estSpatial(cell)
            if infoSpatial[0]:
                score_colonne[j] += 1
                if (score_colonne[j]*10) >= 50 and headers[j] not in liste_attributs_spatiaux.keys():
                    liste_attributs_spatiaux[headers[j]] = [cell, infoSpatial[1]]
    
    return liste_attributs_spatiaux

def recupererAttributsTemporels(headers, df, score_colonne):
    liste_attributs_temporels = {}

    for i in range(10):
        for j in range(len(headers)):
            cell = df.iloc[i, j]
            if not isinstance(cell, str):
                cell = str(cell)

            infoTemporel = estTemporel(cell)
            if infoTemporel[0]:
                score_colonne[j] += 1
                if (score_colonne[j]*10) >= 50 and headers[j] not in liste_attributs_temporels.keys():
                    liste_attributs_temporels[headers[j]] = [cell, infoTemporel[1]]
    return liste_attributs_temporels

def recupererLowGranAndScopeSpa(liste_attributs_spatiaux):
    if len(liste_attributs_spatiaux) > 0:
        le_plus_haut = hierarchie_champs_spa.items()[-1]
        le_plus_bas = hierarchie_champs_spa.items()[0]

        for attribut, champ in liste_attributs_spatiaux.items():
            if hierarchie_champs_spa[champ[1]] < hierarchie_champs_spa[le_plus_bas[0]]:
                le_plus_bas = [champ[1], attribut]
            if hierarchie_champs_spa[champ[1]] > hierarchie_champs_spa[le_plus_haut[0]]:
                le_plus_haut = [champ[1], attribut]
    else:
        le_plus_haut = None
        le_plus_bas = None    
    
    return {'Low granularity': le_plus_bas, 'Scope': le_plus_haut}

def recupererLowGranAndScopeTem(liste_attributs_spatiaux):
    if len(liste_attributs_spatiaux) > 0:
        le_plus_haut = ['QP', 1]
        le_plus_bas = ['regions', 7]

        for attribut, champ in liste_attributs_spatiaux.items():
            if hierarchie_champs_spa[champ[1]] < hierarchie_champs_spa[le_plus_bas[0]]:
                le_plus_bas = [champ[1], attribut]
            if hierarchie_champs_spa[champ[1]] > hierarchie_champs_spa[le_plus_haut[0]]:
                le_plus_haut = [champ[1], attribut]
    else:
        le_plus_haut = None
        le_plus_bas = None    
    
    return {'Low granularity': le_plus_bas, 'Scope': le_plus_haut}

In [12]:
import load_file as lf
import pandas as pd

compteur_datasets = 0

for dataset in datasets['csv']:
    nom_fichier = dataset.split('/')[-1]
    extension = nom_fichier.split('.')[-1]
    compteur_datasets += 1
    score_colonne = {}
    low_gran_and_scope_tem = {}
    low_gran_and_scope_spa = {}
    liste_attributs_spatiaux = {}

    print(f'Traitement du fichier : {dataset[:45]}... | {compteur_datasets}/{len(datasets['csv'])}')
    
    try:
        df = lf.find_type(dataset)[0]
        if len(df.columns.tolist())<5:
            try:
                df = pd.read_csv(dataset, sep=';', encoding='utf-8', nrows=500)
            except:
                df = pd.read_csv(dataset, sep=';', encoding='latin1', nrows=500)
        
        headers = df.columns.tolist()

        for k in range(len(headers)):
            score_colonne[k] = 0
        print(f"En-têtes du fichier {nom_fichier[:10]}...{extension}: {headers}")

    except Exception as e:
        print(f"Erreur lors du chargement du fichier {nom_fichier[:10]}: {e}")
        continue

    liste_attributs_spatiaux = recupererAttributsSpatiaux(headers, df, score_colonne)
    liste_attributs_temporels = recupererAttributsTemporels(headers, df, score_colonne)

    low_gran_and_scope_spa[nom_fichier] = recupererLowGranAndScopeSpa(liste_attributs_spatiaux)
    low_gran_and_scope_tem[nom_fichier] = recupererLowGranAndScopeTem(liste_attributs_spatiaux)

    print(f'Attributs spatiaux retenus : {low_gran_and_scope_spa}')
    print(f'Attributs temporels retenus : {low_gran_and_scope_tem}')

    if compteur_datasets == 10:
        break

Traitement du fichier : Opendata/Landes/Education/fr-en-adresse-et-ge... | 1/52
En-têtes du fichier fr-en-adre...csv: ['numero_uai', 'appellation_officielle', 'denomination_principale', 'patronyme_uai', 'secteur_public_prive_libe', 'adresse_uai', 'lieu_dit_uai', 'boite_postale_uai', 'code_postal_uai', 'localite_acheminement_uai', 'libelle_commune', 'coordonnee_x', 'coordonnee_y', 'EPSG', 'latitude', 'longitude', 'appariement', 'localisation', 'nature_uai', 'nature_uai_libe', 'etat_etablissement', 'etat_etablissement_libe', 'code_departement', 'code_region', 'code_academie', 'code_commune', 'libelle_departement', 'libelle_region', 'libelle_academie', 'position', 'secteur_prive_code_type_contrat', 'secteur_prive_libelle_type_contrat', 'code_ministere', 'libelle_ministere', 'date_ouverture']
Je verifie si spatial
Je verifie le geopoint
Je verifie la latitude
Je verifie la longitude
Je verifie la latitude
Je verifie la longitude
Je verifie l'iris
Je verifie le geopoint
Je verifie la latitu

KeyboardInterrupt: 

In [14]:
import load_file as lf
fichier = 'Opendata/Landes/Education/fr-en-adresse-et-geolocalisation-etablissements-premier-et-second-degre.csv'
score_colonne = {}
try:
    df = lf.find_type(fichier)[0]
    headers = df.columns.tolist()
    for k in range(len(headers)):
        score_colonne[k] = 0
    print(f"En-têtes du fichier {fichier[:10]}: {len(headers)}")
except Exception as e:
    print(f"Erreur lors du chargement du fichier {fichier[:10]}: {e}")
    exit(1)

liste_attributs_spatiaux = recupererAttributsSpatiaux(headers, df, score_colonne)
liste_attributs_temporels = recupererAttributsTemporels(headers, df, score_colonne)
print(f"Attributs spatiaux : {liste_attributs_spatiaux}")
print(f"Attributs temporels : {liste_attributs_temporels}")


En-têtes du fichier Opendata/L: 35
1971-05-24
1971-05-24
1971-05-24
1971-05-24
1971-05-24
1971-05-24
1971-05-24
1972-06-06
1972-06-06
1972-06-06
Attributs spatiaux : {'libelle_commune': ['tremblay-en-france', 'communes'], 'libelle_departement': ['seine-saint-denis', 'departements'], 'libelle_academie': ['créteil', 'communes'], 'localite_acheminement_uai': ['villemomble', 'communes']}
Attributs temporels : {'date_ouverture': ['1971-05-24', 'date']}


## 3. Enregistrer les données dans la classe Dataset

### 3.1 Créer un objet DS_Spatial_Scope